In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

from pprint import pprint
from tavily import TavilyClient
from typing import Dict, Any
from ipywidgets import FileUpload
from IPython.display import display
import base64

In [3]:
client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """search the web for information"""
    return client.search(query)

In [4]:
web_search.invoke("what dish can I make with chicken, rice, and broccoli?")

{'query': 'what dish can I make with chicken, rice, and broccoli?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://familystylefood.com/skillet-chicken-and-rice',
   'title': 'Baked Chicken and Rice with Broccoli - Familystyle Food',
   'content': "This post may contain affiliate links. Please read my disclosure policy.\n\nChicken and rice is a combo that never needs an explanation — it’s just good. This skillet version has earned a permanent spot in my weeknight rotation because everything happens in one ovenproof pan. Brown the chicken on the stovetop, stir in short grain rice, broccoli and Parmesan, then let the oven do the rest. Dinner in 45 minutes, with minimal cleanup. [...] 6. Add the broccoli to the boiling broth and cook one minute. Turn off the heat. Add the rice and ½ cup of the cheese and gently stir them in. Nestle the chicken pieces in the pan, adding in any juices that collected on the plate.\n7. Transfer the pan to the oven a

In [5]:
agent = create_agent(model="gpt-5-nano", tools=[web_search], checkpointer=InMemorySaver(),
                     system_prompt="You are a personal chef that can help me find recipes online based on ingredients I have.")

In [6]:
uploader = FileUpload(accept=".jpg", multiple=False)
display(uploader)

FileUpload(value=(), accept='.jpg', description='Upload')

In [7]:
content_mv = uploader.value[0]["content"]
img_bytes = bytes(content_mv)
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [8]:
multipart_message = HumanMessage(content=[
    {"type": "text", "text": "Tell me what food found in this image."},
    {"type": "image", "base64": img_b64, "mime_type": "image/jpeg"}
])

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke({"messages": [multipart_message]}, config=config)
pprint(response["messages"][-1].content)

('Here are the foods I can identify in the fridge image, organized by type:\n'
 '\n'
 'Fruits\n'
 '- Strawberries\n'
 '- Raspberries\n'
 '- Blackberries (in the berry bowl)\n'
 '- Oranges (bottom drawer)\n'
 '- Lemons (bottom drawer)\n'
 '- Possibly other citrus in bottles, but primary citrus visible: oranges and '
 'lemons\n'
 '\n'
 'Vegetables\n'
 '- Tomatoes (regular and cherry tomatoes)\n'
 '- Bell peppers (orange, red, yellow)\n'
 '- Carrots\n'
 '- Mushrooms\n'
 '- Broccoli\n'
 '- Lettuce/greens\n'
 '- Green beans or okra (long green pods)\n'
 '- Possibly a head of cabbage or other greens tucked in\n'
 '\n'
 'Proteins\n'
 '- Eggs (large carton on the door)\n'
 '- Roasted chicken (on a plate)\n'
 '- Sliced ham (on a white tray)\n'
 '- Raw hamburger patties (on a plate)\n'
 '- Sausages/hot dogs (on the shelf)\n'
 '- Possibly other meat items tucked in the back shelves\n'
 '\n'
 'Dairy / beverages / condiments\n'
 '- Milk jug\n'
 '- Various bottled drinks (orange juice, red juice, ot

In [9]:
response = agent.invoke({"messages": [HumanMessage(content="are there apples in the fridge?")]}, config=config)
pprint(response["messages"][-1].content)

('No. Apples aren’t visible in this fridge image. I do see oranges and lemons '
 'in the bottom drawers, plus various veggies, eggs, and beverages. Want '
 'recipe ideas using what’s here?')


In [10]:
question = HumanMessage(content="what dish can I make with food found in my fridge?")
response = agent.invoke({"messages": [question]}, config=config)
pprint(response["messages"][-1].content)

("Nice! Here are several dish ideas you can make with what's visible in your "
 'fridge. I’ve kept them simple and used ingredients I can see: eggs, chicken, '
 'ham, beef burgers, sausage, peppers, mushrooms, tomatoes, carrots, broccoli, '
 'greens, potatoes, and some citrus.\n'
 '\n'
 '1) Veggie-packed omelette or frittata\n'
 '- Main ingredients: eggs, bell peppers, mushrooms, tomatoes, greens '
 '(lettuce/greens).\n'
 '- Quick steps: Sauté sliced peppers and mushrooms in olive oil until tender. '
 'Beat 3–4 eggs with salt and pepper, pour over veggies, add chopped tomatoes '
 'and greens. Cook until set, finishing under the broiler if you like a golden '
 'top. Serves 1–2.\n'
 '\n'
 '2) One-pan chicken and roasted veggies\n'
 '- Main ingredients: chicken, potatoes, carrots, bell peppers, broccoli, '
 'lemon (optional).\n'
 '- Quick steps: Toss chopped veggies with olive oil, salt, pepper, and '
 'garlic/herbs. Roast on a sheet pan at 425°F (220°C) for 25–35 minutes. If '
 'you have